<a href="https://colab.research.google.com/github/Noors-lab/VigIQ-Vigilance-Intelligence-/blob/main/full%20script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import os
from google.colab import drive

drive.mount('/content/drive')

base = "/content/drive/MyDrive/shoplifting/vigiq_github"
dirs = [
    f"{base}/src",
    f"{base}/models",
    f"{base}/notebooks",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Structure created.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Structure created.


In [18]:
requirements = """torch>=2.0.0
torchvision>=0.15.0
ultralytics>=8.0.0
opencv-python>=4.8.0
numpy>=1.24.0
fastapi>=0.100.0
uvicorn>=0.23.0
python-multipart>=0.0.6
aiofiles>=23.0.0
pyngrok>=6.0.0
scikit-learn>=1.3.0
"""

with open(f"{base}/requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt saved ✓")

requirements.txt saved ✓


In [19]:
utils = '''
import numpy as np

# Keypoint indices — YOLOv11 pose
NOSE           = 0
LEFT_SHOULDER  = 5
RIGHT_SHOULDER = 6
LEFT_ELBOW     = 7
RIGHT_ELBOW    = 8
LEFT_WRIST     = 9
RIGHT_WRIST    = 10
LEFT_HIP       = 11
RIGHT_HIP      = 12
LEFT_KNEE      = 13
RIGHT_KNEE     = 14

def normalize_keypoints(kps):
    """
    Normalize keypoints relative to body center and scale.
    kps: numpy array of shape (17, 2)
    Returns: flattened list of 34 normalized values
    """
    left_hip   = kps[LEFT_HIP]
    right_hip  = kps[RIGHT_HIP]
    center     = (left_hip + right_hip) / 2

    l_shoulder = kps[LEFT_SHOULDER]
    r_shoulder = kps[RIGHT_SHOULDER]
    shoulder_c = (l_shoulder + r_shoulder) / 2

    scale = np.linalg.norm(shoulder_c - center)
    if scale < 1e-6:
        scale = 1.0

    normalized = (kps - center) / scale
    return normalized.flatten().tolist()
'''

with open(f"{base}/src/utils.py", "w") as f:
    f.write(utils)

print("src/utils.py saved ✓")

src/utils.py saved ✓


In [20]:
rules = '''
import numpy as np
import torch

NOSE          = 0
LEFT_WRIST    = 9
RIGHT_WRIST   = 10

def apply_rules(kps_sequence, lstm_model, X_mean, X_std, device):
    """
    Apply rule layer on top of LSTM.
    Returns (alert: bool, reason: str, confidence: float)
    """
    if len(kps_sequence) < 10:
        return False, "insufficient frames", 0.0

    # ── Rule 1: LSTM confidence ──
    seq = np.array(kps_sequence[-50:], dtype=np.float32)
    while len(seq) < 50:
        seq = np.vstack([seq, np.zeros((1, 34))])
    seq = (seq - X_mean) / (X_std + 1e-8)
    tensor = torch.tensor(seq, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        logit = lstm_model(tensor)
        confidence = torch.sigmoid(logit).item()

    if confidence < 0.80:
        return False, f"low confidence ({confidence:.2f})", confidence

    # ── Rule 2: Wrist destination check ──
    recent = np.array(kps_sequence[-15:])
    if len(recent) >= 5:
        lw_x = recent[:, LEFT_WRIST*2]
        rw_x = recent[:, RIGHT_WRIST*2]
        wrist_check = (abs(lw_x[-1] - lw_x[0]) > 0.1 or
                       abs(rw_x[-1] - rw_x[0]) > 0.1)
    else:
        wrist_check = False

    if not wrist_check:
        return False, "wrist not moving toward body", confidence

    # ── Rule 3: Sustained motion ──
    if len(kps_sequence) < 20:
        return False, "motion not sustained", confidence

    # ── Rule 4: Head rotation ──
    nose_x = np.array(kps_sequence[-20:])[:, NOSE*2]
    head_rotating = np.var(nose_x) > 0.01

    # ── Rule 5: Hooded person ──
    nose_pos = np.array(kps_sequence[-10:])[:, NOSE*2]
    hooded   = np.mean(np.abs(nose_pos)) < 0.05

    if hooded and confidence >= 0.65:
        return True, "HOODED PERSON + suspicious motion", confidence

    if head_rotating:
        return True, "suspicious motion + head rotation", confidence

    return True, "suspicious motion detected", confidence
'''

with open(f"{base}/src/rules.py", "w") as f:
    f.write(rules)

print("src/rules.py saved ✓")

src/rules.py saved ✓


In [21]:
pipeline = '''
import cv2
import numpy as np
import torch
import torch.nn as nn
from collections import defaultdict
from ultralytics import YOLO
from datetime import datetime
import os
import json

from utils import normalize_keypoints
from rules import apply_rules

# ─── LSTM Model ───
class ShopliftingLSTM(nn.Module):
    def __init__(self, input_size=34, hidden_size=256,
                 num_layers=3, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout,
            bidirectional=True
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.classifier(out).squeeze()

def load_models(model_path, mean_path, std_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    lstm = ShopliftingLSTM().to(device)
    lstm.load_state_dict(torch.load(model_path, map_location=device))
    lstm.eval()

    X_mean = np.load(mean_path)[0]
    X_std  = np.load(std_path)[0]
    yolo   = YOLO("yolo11n-pose.pt")

    return lstm, yolo, X_mean, X_std, device

def extract_clip(frame_buffer, fps, output_path):
    if not frame_buffer:
        return
    h, w = frame_buffer[0].shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))
    for frame in frame_buffer:
        out.write(frame)
    out.release()

def run_pipeline(source, lstm, yolo, X_mean, X_std, device,
                 output_dir="vigiq_alerts", max_frames=None):

    os.makedirs(output_dir, exist_ok=True)

    cap         = cv2.VideoCapture(source)
    fps         = cap.get(cv2.CAP_PROP_FPS) or 30.0
    BUFFER_SIZE = int(fps * 10)

    frame_buffer   = []
    kps_sequences  = defaultdict(list)
    alerted_ids    = set()
    alert_log      = []
    frame_idx      = 0
    SAMPLE_EVERY   = 3

    log_path = os.path.join(output_dir, "alert_log.json")

    # Load existing alerts if any
    if os.path.exists(log_path):
        with open(log_path) as f:
            alert_log = json.load(f)

    print(f"VigIQ Pipeline started — Source: {source}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if max_frames and frame_idx >= max_frames:
            break

        frame_buffer.append(frame.copy())
        if len(frame_buffer) > BUFFER_SIZE:
            frame_buffer.pop(0)

        if frame_idx % SAMPLE_EVERY == 0:
            results = yolo.track(frame, persist=True, verbose=False)

            if (results[0].boxes.id is not None and
                    results[0].keypoints is not None):

                track_ids = results[0].boxes.id.cpu().numpy().astype(int)
                keypoints = results[0].keypoints.xy.cpu().numpy()

                for tid, kps in zip(track_ids, keypoints):
                    if kps.shape[0] != 17:
                        continue

                    normalized = normalize_keypoints(kps)
                    kps_sequences[tid].append(normalized)

                    if len(kps_sequences[tid]) > 50:
                        kps_sequences[tid].pop(0)

                    if tid in alerted_ids:
                        continue

                    if frame_idx % 10 != 0:
                        continue

                    alert, reason, conf = apply_rules(
                        kps_sequences[tid], lstm, X_mean, X_std, device
                    )

                    if alert:
                        timestamp  = frame_idx / fps
                        clip_name  = f"alert_{tid}_{int(timestamp)}s.mp4"
                        clip_path  = os.path.join(output_dir, clip_name)

                        extract_clip(frame_buffer, fps, clip_path)

                        alert_entry = {
                            "person_id":  int(tid),
                            "reason":     reason,
                            "confidence": round(float(conf), 2),
                            "timestamp":  round(timestamp, 1),
                            "clip":       clip_name,
                            "time":       datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                            "status":     "pending"
                        }
                        alert_log.append(alert_entry)
                        alerted_ids.add(tid)

                        with open(log_path, "w") as f:
                            json.dump(alert_log, f, indent=2)

                        print(f"ALERT — Person {tid} | {reason} | {conf:.2f} | {timestamp:.1f}s")

        frame_idx += 1

    cap.release()
    print(f"Pipeline finished. Alerts: {len(alert_log)}")
    return alert_log
'''

with open(f"{base}/src/pipeline.py", "w") as f:
    f.write(pipeline)

print("src/pipeline.py saved ✓")

src/pipeline.py saved ✓


In [22]:
dashboard_code = '''
import os
import json
import threading
import time
from fastapi import FastAPI
from fastapi.responses import HTMLResponse, FileResponse
import uvicorn

ALERTS_DIR = "vigiq_alerts"
os.makedirs(ALERTS_DIR, exist_ok=True)

app = FastAPI()

@app.get("/api/alerts")
def get_alerts():
    log_path = os.path.join(ALERTS_DIR, "alert_log.json")
    if not os.path.exists(log_path):
        return []
    with open(log_path) as f:
        return json.load(f)

@app.post("/api/alerts/{alert_id}/confirm")
def confirm_alert(alert_id: int):
    return update_status(alert_id, "confirmed")

@app.post("/api/alerts/{alert_id}/dismiss")
def dismiss_alert(alert_id: int):
    return update_status(alert_id, "dismissed")

def update_status(alert_id, status):
    log_path = os.path.join(ALERTS_DIR, "alert_log.json")
    with open(log_path) as f:
        alerts = json.load(f)
    for alert in alerts:
        if alert.get("person_id") == alert_id:
            alert["status"] = status
    with open(log_path, "w") as f:
        json.dump(alerts, f, indent=2)
    return {"success": True}

@app.get("/api/clip/{clip_name}")
def get_clip(clip_name: str):
    path = os.path.join(ALERTS_DIR, clip_name)
    if os.path.exists(path):
        return FileResponse(path, media_type="video/mp4")
    return {"error": "not found"}

@app.get("/", response_class=HTMLResponse)
def index():
    with open("src/dashboard.html") as f:
        return f.read()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open(f"{base}/src/dashboard.py", "w") as f:
    f.write(dashboard_code)

print("src/dashboard.py saved ✓")

src/dashboard.py saved ✓
